In [18]:
from pathlib import Path
import os

model_path = Path(
    "/data/models/gemma-3-1b-it"
)

print("Putanja postoji:", model_path.exists())
print("Jeste direktorijum:", model_path.is_dir())
print("/lustre postoji:", Path("/lustre").exists())
print("Jupyter cwd:", os.getcwd())

if model_path.exists():
    print("\nFajlovi:")
    for path in sorted(model_path.iterdir()):
        print(path.name)

Putanja postoji: True
Jeste direktorijum: True
/lustre postoji: True
Jupyter cwd: /home/mls01/scripts/model

Fajlovi:
.cache
.gitattributes
README.md
added_tokens.json
config.json
generation_config.json
model.safetensors
special_tokens_map.json
tokenizer.json
tokenizer.model
tokenizer_config.json


In [19]:
!find /lustre -maxdepth 7 -type d -name "gemma-3-1b-it" 2>/dev/null

In [20]:
import os
from pathlib import Path

os.environ["USER"] = "mls01"
os.environ["LOGNAME"] = "mls01"
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/home/mls01/.cache/torchinductor"
os.environ["TRITON_CACHE_DIR"] = "/home/mls01/.cache/triton"
os.environ["XDG_CACHE_HOME"] = "/home/mls01/.cache"

Path(os.environ["TORCHINDUCTOR_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["TRITON_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)

print("Cache konfiguracija: OK")


Cache konfiguracija: OK


In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=True,
)

model.eval()

print("Gemma 3 1B IT je uspešno učitan.")
print("Model type:", model.config.model_type)
print("Device:", next(model.parameters()).device)


Gemma 3 1B IT je uspešno učitan.
Model type: gemma3_text
Device: cuda:0


## Priprema i podela dataseta

Ova sekcija samo učitava, čisti i deli `complete_dataset.jsonl`. Nema Gemma inferencije, nema klasifikacionog prompta, nema metrika.

In [ ]:
import pandas as pd

DATASET_PATH = "/home/mls01/data/complete_dataset.jsonl"

df = pd.read_json(DATASET_PATH, lines=True)
print("Dataset učitan:", DATASET_PATH)

In [ ]:
print("Broj redova:", len(df))
print("Broj kolona:", df.shape[1])
print("\nSpisak kolona:")
print(list(df.columns))

for col in ["prompt_harm_label", "response_harm_label", "response_refusal_label"]:
    print(f"\nDistribucija kolone '{col}':")
    print(df[col].value_counts(dropna=False))

In [ ]:
df["response"] = df["response"].fillna("")
print("Prazni odgovori (response == \"\"):", (df["response"] == "").sum())

In [ ]:
df["final_label"] = "unharmful"
harmful_mask = (
    (df["prompt_harm_label"] == "harmful")
    | (df["response_harm_label"] == "harmful")
    | (df["response_refusal_label"] == "refusal")
)
df.loc[harmful_mask, "final_label"] = "harmful"

counts = df["final_label"].value_counts()
percentages = df["final_label"].value_counts(normalize=True) * 100
for label in counts.index:
    print(f"{label}: {counts[label]} redova ({percentages[label]:.2f}%)")

In [ ]:
assert "row_id" in df.columns, "Kolona 'row_id' ne postoji."
assert df["row_id"].is_unique, "Postoje duplikati u 'row_id'."
assert df["original_idx"].notna().all(), "Postoje prazne vrednosti u 'original_idx'."

label_counts_per_group = df.groupby("original_idx")["final_label"].nunique()
inconsistent_groups = label_counts_per_group[label_counts_per_group > 1]

if len(inconsistent_groups) > 0:
    raise ValueError(
        f"Nekonzistentne final_label vrednosti unutar original_idx grupa: "
        f"{list(inconsistent_groups.index)}"
    )

print("Validacija OK: row_id jedinstven, original_idx popunjen, final_label konzistentan po grupama.")

In [ ]:
WORK_COLUMNS = [
    "row_id", "original_idx", "prompt", "response",
    "prompt_harm_label", "response_harm_label", "response_refusal_label",
    "final_label", "language", "adversarial", "augmentation_type",
    "encoding_type", "subcategory", "response_truncated",
]

work_df = df[WORK_COLUMNS].copy()
print("work_df shape:", work_df.shape)
work_df.head()

In [ ]:
from sklearn.model_selection import train_test_split

groups = (
    work_df[["original_idx", "final_label"]]
    .drop_duplicates("original_idx")
    .reset_index(drop=True)
)

train_ids, temp_ids = train_test_split(
    groups["original_idx"],
    test_size=0.2,
    stratify=groups["final_label"],
    random_state=42,
)

temp_labels = groups.set_index("original_idx").loc[temp_ids, "final_label"]

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42,
)

print("Train original_idx grupa:", len(train_ids))
print("Validation original_idx grupa:", len(val_ids))
print("Test original_idx grupa:", len(test_ids))

In [ ]:
split_map = {}
split_map.update({idx: "train" for idx in train_ids})
split_map.update({idx: "validation" for idx in val_ids})
split_map.update({idx: "test" for idx in test_ids})

work_df["split"] = work_df["original_idx"].map(split_map)

train_df = work_df[work_df["split"] == "train"].reset_index(drop=True)
val_df = work_df[work_df["split"] == "validation"].reset_index(drop=True)
test_df = work_df[work_df["split"] == "test"].reset_index(drop=True)

print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("test_df:", test_df.shape)

In [ ]:
train_ids_set = set(train_df["original_idx"])
val_ids_set = set(val_df["original_idx"])
test_ids_set = set(test_df["original_idx"])

assert not (train_ids_set & val_ids_set), "Preklapanje original_idx između train i validation."
assert not (train_ids_set & test_ids_set), "Preklapanje original_idx između train i test."
assert not (val_ids_set & test_ids_set), "Preklapanje original_idx između validation i test."

all_ids = train_ids_set | val_ids_set | test_ids_set
assert all_ids == set(work_df["original_idx"]), "Svaki original_idx mora pripadati tačno jednom splitu."

total_row_ids = pd.concat([train_df["row_id"], val_df["row_id"], test_df["row_id"]])
assert total_row_ids.is_unique, "Postoje duplikati row_id nakon podele."
assert set(total_row_ids) == set(work_df["row_id"]), "Neki row_id su izgubljeni nakon podele."

assert len(train_df) + len(val_df) + len(test_df) == len(work_df), "Ukupan broj redova se ne poklapa."

print("Završna validacija OK: nema preklapanja, nema izgubljenih/dupliranih row_id.")

In [ ]:
for name, split_df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    n_groups = split_df["original_idx"].nunique()
    n_rows = len(split_df)
    print(f"\n=== {name} ===")
    print(f"Broj original_idx grupa: {n_groups}")
    print(f"Broj redova: {n_rows}")

    group_labels = split_df.drop_duplicates("original_idx")["final_label"]
    group_counts = group_labels.value_counts()
    group_pct = group_labels.value_counts(normalize=True) * 100
    print("Labele po original_idx grupama:")
    for label in group_counts.index:
        print(f"  {label}: {group_counts[label]} ({group_pct[label]:.2f}%)")

    row_counts = split_df["final_label"].value_counts()
    row_pct = split_df["final_label"].value_counts(normalize=True) * 100
    print("Labele po redovima:")
    for label in row_counts.index:
        print(f"  {label}: {row_counts[label]} ({row_pct[label]:.2f}%)")